# 0: Preprocessing

This notebook preprocesses the doreco data and the TES taken from Beekhuizen (2024).
1. Corpus Data
    - Selecting corpora containing narrative data
    - Extracting narrative and non-narrative (else) corpora
    - Token counts for total, narrative and else corpora
2. Translation Equivalencies
    - Selecting TES language with corpora containing narrative data
    - Merging TES stems using Beekhuizen (2024)'s procedure

References:
- Beekhuizen, B. (2024). Discursive distinctiveness explains lexical differences between languages. The Evolution of Language: Proceedings of the 15th International Conference (Evolang XV). doi:10.17617/2.3587960.

In [1]:
import dill
import json
import csv

import networkx as nx
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from gensim.models import Word2Vec
from collections import defaultdict, Counter
from itertools import combinations, chain
from editdistance import eval as ed
from scipy.stats import entropy
from numpy.linalg import norm

In [2]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /home/shore/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/shore/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/shore/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
# declaring paths and filenames

raw_data_path = '../data/raw/'
processed_data_path = '../data/processed/'

raw_corpus_fn = raw_data_path + 'doreco_corpus.p'
raw_tes_fn = raw_data_path + 'tes/tes_'
metadata_fn = raw_data_path + 'doreco_files_metadata.csv'

corpus_fn = processed_data_path + 'corpus.pkl'
corpus_narr_fn = processed_data_path + 'corpus_narr.pkl'
corpus_else_fn = processed_data_path + 'corpus_else.pkl'
token_counts_fn = processed_data_path + 'token_counts.csv'

tes_fn = processed_data_path + 'tes.pkl'
merged_tes_fn = processed_data_path + 'merged_tes.pkl'

## Corpus Data

In [4]:
def select_genres(corpus, files_metadata, genres):
    """ Returns a sub-corpus containing only the given genres. Taken from Beekhuizen (2024)."""
    metadata = csv.DictReader(open(files_metadata), delimiter='\t')
    metadata = {(x['doculect'][:4],x['file']): (x['genre'],int(x['tokens'])) for x in metadata}
    genre_counts = Counter()
    new_corpus = defaultdict(lambda : {})
    for f in corpus:
        for fi in corpus[f]:
            fx = '_'.join(fi.split('_')[2:])
            genre = metadata[f[:4],fx][0]
            genre_counts[genre] += 1
            if genre in genres:
                new_corpus[f][fi] = corpus[f][fi]
    #print(genre_counts.most_common(10))
    return new_corpus

In [5]:
EXCLUDED_LANGS = {'ligh1234.csv', 'stan1290.csv', 'warl1254.csv'} # languages without TES files (no narr data)
GENRES = {'personal narrative', 'traditional narrative', 'conversation', 'procedural'}
NARR_GENRES = {'personal narrative', 'traditional narrative'}
ELSE_GENRES = {'conversation', 'procedural'}

In [6]:
# loading corpus, selecting retained languages, creating genre-specific sub-corpora
raw_corpus = select_genres(dill.load(open(raw_corpus_fn, 'rb')), metadata_fn, GENRES)
corpus = {lang.split('.')[0]: entry for lang, entry in raw_corpus.items() if lang not in EXCLUDED_LANGS}
corpus_narr = select_genres(corpus, metadata_fn, NARR_GENRES)
corpus_else = select_genres(corpus, metadata_fn, ELSE_GENRES)

In [7]:
# saving everything
dill.dump(corpus, open(corpus_fn, 'wb'))
dill.dump(corpus_narr, open(corpus_narr_fn, 'wb'))
dill.dump(corpus_else, open(corpus_else_fn, 'wb'))

### Token Counts
todo:
- compute counts from removed langs

In [8]:
def token_count(langcorp):
    count = 0
    for document in langcorp:
        for sentence in langcorp[document]:
            count += len(langcorp[document][sentence]['spc'])
    return count

In [9]:
counts = {'lang': [], 'n_tokens': [], 'n_tokens_narr': [], 'n_tokens_else': []}
for lang in corpus:
    counts['lang'].append(lang)
    counts['n_tokens'].append(token_count(corpus[lang]))
    counts['n_tokens_narr'].append(token_count(corpus_narr[lang]))
    counts['n_tokens_else'].append(token_count(corpus_else[lang]))
counts_df = pd.DataFrame(counts)
counts_df

,lang,n_tokens,n_tokens_narr,n_tokens_else
0,anal1239,20041,17932,2109
1,apah1238,18726,13156,5570
2,arap1274,35839,35008,831
3,bain1259,22733,7532,15201
4,beja1238,38121,35748,2373
5,bora1263,53905,48218,5687
6,cabe1245,8472,3183,5289
7,cash1254,21875,21875,0
8,dolg1241,34863,31774,3089
9,even1259,19877,19667,210


In [10]:
counts_df.to_csv(token_counts_fn)

## Translation Equivalencies
todo:
- replace tokenization with spacy

In [11]:
tes = {lang.split('.')[0]: dill.load(open(raw_tes_fn + lang + '.csv.p', 'rb')) for lang in corpus if lang not in EXCLUDED_LANGS}

In [12]:
dill.dump(tes, open(tes_fn, 'wb'))

### Merging TES

In [13]:
### EXTRACTING FREE TRANSLATIONS & TRAINING W2V

def preprocess_text(text):
    #stop_words = set(stopwords.words('english'))
    # replace with spacy
    tokens = word_tokenize(text.lower())
    #tokens = spc(text.lower())
    tokens = [word for word in tokens if word.isalpha()]
    return tokens

fts = [preprocess_text(corpus[lang][doc][sen]['ft']) for lang in corpus for doc in corpus[lang] for sen in corpus[lang][doc]]

wv = Word2Vec(sentences=fts, min_count=1)

In [14]:
### MERGING FUNCTIONS

def JSD(P, Q):
    """Taken from Beekhuizen (2024)."""
    _P = P / norm(P, ord=1)
    _Q = Q / norm(Q, ord=1)
    _M = 0.5 * (_P + _Q)
    return 0.5 * (entropy(_P, _M) + entropy(_Q, _M))

def merge_similar_tes_within(tes, te_words):
    """Taken from Beekhuizen (2024)."""
    merge = nx.Graph()
    merge.add_nodes_from(tes)
    tes_new, te_words_new = {}, {}
    for t1, t2 in combinations(tes,2):
        if t1 in t2 or t2 in t1 or ed(t1,t2) <= 1:
            merge.add_edge(t1,t2)
    for c in nx.connected_components(merge):
        best_c = max(c, key= lambda k : len(tes[k]))
        tes_new[best_c] = list(chain(*[tes[ci] for ci in c]))
        te_words_new[best_c] = sum([te_words[ci] for ci in c], Counter())
        #print('>>> MERGED', c)
    return tes_new, te_words_new

def merge_similar_tes_across(tes, te_words):
    """Taken from Beekhuizen (2024)."""
    merge_markers = nx.Graph()
    te_marker_pairs = [(te, marker) for te in tes for marker in tes[te] if len(tes[te][marker]) >= 5]
    for (tei,mrki),(tej,mrkj) in combinations(te_marker_pairs,2):
        if tei==tej: continue
        if set(te_words[tei][mrki]) & set(te_words[tej][mrkj]) != set():
            mrkic, mrkjc = mrki.strip('^$'), mrkj.strip('^$')
            minlen = min(len(mrkic),len(mrkjc))
            wi = te_words[tei][mrki]
            wj = te_words[tej][mrkj]
            jsd = (JSD(*zip(*[[wi[x],wj[x]] for x in sorted(set(wi)|set(wj))])))
            try:
                cos = (wv.wv.similarity(tei,tej))
            except:
                print('cannot compute similarity', tei, tej)
                continue
            jac = len(set(wi)&set(wj))/len(set(wi)|set(wj))
            form = mrkic[:minlen] == mrkjc[:minlen] or mrkic[-minlen:] == mrkjc[-minlen:]
            if cos >= 2/3 and jsd <= 2/3 and form:
                #print((tei,mrki),(tej,mrkj),'JSD=%.2f COS=%.2f JAC=%.2f FORM=%d' % (jsd, cos, jac,form))
                merge_markers.add_edge((tei,mrki), (tej,mrkj))
    for c in nx.connected_components(merge_markers):
        new_mrk = min([mrk for te,mrk in c],key = len)
        for te,mrk in c:
            tes[te][new_mrk] = tes[te][mrk]
            te_words[te][new_mrk] = te_words[te][mrk]
            if mrk != new_mrk: del tes[te][mrk], te_words[te][mrk]
    return tes, te_words

def merge_similar_tes(langtes):
    """Taken from Beekhuizen (2024)."""
    tes, te_words = langtes
    for k,v in tes.items():
        tes[k], te_words[k] = merge_similar_tes_within(v, te_words[k])
    #
    tes, te_words = merge_similar_tes_across(tes, te_words)
    return tes, te_words

In [15]:
# generating merged tes
merged_tes = dict()
for lang in tes:
    merged_tes[lang] = merge_similar_tes(tes[lang])

cannot compute similarity give alm
cannot compute similarity go buttock
cannot compute similarity u qururu
cannot compute similarity use hoocak
cannot compute similarity go hoocak
cannot compute similarity berry blueberry
cannot compute similarity pig jarisdosegi
cannot compute similarity tongse thongshe
cannot compute similarity go eld
cannot compute similarity sister eld
cannot compute similarity reason helper
cannot compute similarity cross crosse
cannot compute similarity bat eare
cannot compute similarity fox eare
cannot compute similarity give alm
cannot compute similarity catch prawn
cannot compute similarity brother eld
cannot compute similarity happen eld
cannot compute similarity sister eld
cannot compute similarity most eld
cannot compute similarity finish graduate
cannot compute similarity day 9th
cannot compute similarity month 9th
cannot compute similarity story telltar
cannot compute similarity 9th ninth
cannot compute similarity tamale tortilla
cannot compute similarity

In [16]:
dill.dump(merged_tes, open(merged_tes_fn, 'wb'))

In [11]:
# token count for language corpora
def token_count(langcorp):
    count = 0
    for document in langcorp:
        for sentence in langcorp[document]:
            count += len(langcorp[document][sentence]['spc'])
    return count

In [61]:
token_count(corpus['yura1255'])

8128